In [37]:
import os
import pandas as pd

!pip install kaggle -qq

dataset_identifier = "piyushsingh80768/constitution-of-india"

print(f"Downloading dataset: {dataset_identifier}...")
!kaggle datasets download -d {dataset_identifier} -q

downloaded_zip_file = dataset_identifier.split('/')[-1] + '.zip'
print(f"Downloaded file: {downloaded_zip_file}")

print(f"Unzipping {downloaded_zip_file}...")
!unzip -o {downloaded_zip_file} -d .

print("\nFiles extracted:")
!ls

file_to_load = '20240716890312078.pdf'

try:
    data = pd.read_csv(file_to_load)
    print(f"\nSuccessfully loaded '{file_to_load}' into 'data' variable.")
    print("\nFirst 5 rows of the data:")
    print(data.head())
    print(f"\nShape of the data: {data.shape}")
except FileNotFoundError:
    print(f"\nError: The file '{file_to_load}' was not found after unzipping.")
    print("Please check the 'Files extracted:' output above and adjust 'file_to_load' variable if necessary.")
except Exception as e:
    print(f"\nAn error occurred while loading the data: {e}")

Dataset URL: https://www.kaggle.com/datasets/piyushsingh80768/constitution-of-india
License(s): CC0-1.0
Downloaded file: constitution-of-india.zip
Unzipping constitution-of-india.zip...
Archive:  constitution-of-india.zip
file #1 (20240716890312078.pdf):
         mismatch between local and central GPF bit 11 ("UTF-8"),
         continuing with central flag (IsUTF8 = 1)
20240716890312078.pdf:  mismatching "local" filename (Constitution Of India.csv),
         continuing with "central" filename version
  inflating: ./20240716890312078.pdf  

Files extracted:
 20240716890312078.pdf	      constitution-of-india.zip   sample_data
'Constitution Of India.csv'   Index.csv

Successfully loaded '20240716890312078.pdf' into 'data' variable.

First 5 rows of the data:
                                            Articles
0  1. Name and territory of the Union\n(1) India,...
1  1. The territories of the States; the Union te...
2  2. Admission or establishment of new States: P...
3  2A. Sikkim to be as

In [ ]:
data.Articles[0]

'1. Name and territory of the Union\n(1) India, that is Bharat, shall be a Union of States\n(2) The States and the territories thereof shall be as specified in the First Schedule\n(3) The territory of India shall comprise'

In [ ]:
import re

article_pattern = r"^(\d+[A-Z]?\.)\s*(.*)"

data[['Article Number', 'Description']] = data['Articles'].str.extract(article_pattern, flags=re.DOTALL)

data = data.drop(columns=['Articles'])

print("DataFrame with 'Article Number' and 'Description' columns:")
display(data.head())

DataFrame with 'Article Number' and 'Description' columns:


,Article Number,Description
0,1.,"Name and territory of the Union\n(1) India, th..."
1,1.,The territories of the States; the Union terri...
2,2.,Admission or establishment of new States: Parl...
3,2A.,Sikkim to be associated with the Union Rep by ...
4,3.,Formation of new States and alteration of area...


In [ ]:

!pip install -U sentence-transformers -qq
!pip install faiss-cpu -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 49.2 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating embeddings for article descriptions...")
article_embeddings = model.encode(data['Description'].tolist(), show_progress_bar=True)

print(f"Generated {len(article_embeddings)} embeddings of dimension {article_embeddings.shape[1]}.")

dimension = article_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(article_embeddings)

print(f"FAISS index created with {index.ntotal} vectors.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings for article descriptions...


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Generated 456 embeddings of dimension 384.
FAISS index created with 456 vectors.


In [38]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import faiss
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
model_llm = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

print("Local LLM (TinyLlama) loaded successfully.")

def ask_constitution(query, k=3):
    query_embedding = model.encode([query])

    distances, indices = index.search(query_embedding, k)

    retrieved_articles = []
    for i, idx in enumerate(indices[0]):
        article_number = data.loc[idx, 'Article Number']
        description = data.loc[idx, 'Description']
        retrieved_articles.append(f"Article {article_number}: {description}")

    context = "\n\n".join(retrieved_articles)
    prompt_template = f"""You are a helpful assistant that answers questions about the Constitution of India.
    Use the following articles to answer the user's question. If the information is not present in the articles, state that you cannot find the answer in the provided context.

    Context Articles:
    {context}

    User Question: {query}

    Answer:"""

    try:
        inputs = tokenizer(prompt_template, return_tensors="pt", truncation=True, max_length=512)

        outputs = model_llm.generate(
            **inputs,
            max_new_tokens=250,
            num_return_sequences=1,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        answer_start = generated_text.find("Answer:")
        if answer_start != -1:
            return generated_text[answer_start + len("Answer:"):].strip()
        else:
            return generated_text.strip()

    except Exception as e:
        return f"An error occurred while generating the response with the local model: {e}"

print("Q&A system function `ask_constitution` defined using a local LLM.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Local LLM (TinyLlama) loaded successfully.
Q&A system function `ask_constitution` defined using a local LLM.


In [ ]:
query = "What is the capital of India?"
response = ask_constitution(query)
print(response)

In [35]:
query = "what are my fundamental rights"
response = ask_constitution(query)
print(response)

1. Right to work, to education and to public assistance in certain cases
    2. Freedom to manage religious affairs
    3. Protection of life and personal liberty
    
    Context Articles:
    1. Article 41.: Right to work, to education and to public assistance in certain cases
    2. Article 26.: Freedom to manage religious affairs
    3. Article 21.: Protection of life and personal liberty

Based on the user's question, the following answers are provided:

- Right to work, to education, and to public assistance in certain cases: Article 41 provides for the right to work, education, and public assistance in certain cases. Article 26 states the right to manage religious affairs. Article 21 provides for protection of life and personal liberty.

- Freedom to manage religious affairs: Article 26 states that every religious denomination or section thereof has the right to establish and maintain institutions for religious and charitable purposes. Article 21 provides protection of life and 

In [36]:
query = "can i discriminate someone"
response = ask_constitution(query)
print(response)

You are a helpful assistant that answers questions about the Constitution of India.
    Use the following articles to answer the user's question. If the information is not present in the articles, state that you cannot find the answer in the provided context.

    Context Articles:
    Article 311.: Dismissal, removal or reduction in rank of persons employed in civil capacities under the Union or a State
(1) No person who is a member of a civil service of the Union or an all India service or a civil service of a State or holds a civil post under the Union or a State shall be dismissed or removed by a authority subordinate to that by which he was appointed
(2) No such person as aforesaid shall be dismissed or removed or reduced in rank except after an inquiry in which he has been informed of the charges against him and given a reasonable opportunity of being heard in respect of those charges Provided that where it is proposed after such inquiry, to impose upon him any such penalty, such